# Stage 4b/4c — Attribution and scoring

**Attach:** `sarvam-diar-code`, `sarvam-diar-stage3`, `sarvam-diar-asr-indic`,
`sarvam-diar-asr-whisper`.
**Settings:** accelerator **None** — this stage is CPU-only and finishes in
seconds. Running it on a GPU session spends quota on nothing.

Stage 4a produced words with timestamps and no speaker. This stage crosses those
words with **one** diarization hypothesis at a time and writes one directory per
`(asr, diar)` condition. Because the words are identical across conditions, a
cpWER difference between two of them is attributable to the labelling — which is
the entire reason ASR and attribution are separate stages.

No audio is needed here, so the audio dataset stays detached.

### The four rules, and why each one is a choice

1. **Maximum overlap.** A word goes to the turn sharing the most time with it.
   Assigning by midpoint is cheaper and throws away exactly the information that
   matters on words straddling a boundary. Equal overlap breaks toward the
   earlier turn, so the output is deterministic.
2. **Orphans are kept, not dropped.** A word landing where the diarizer heard
   nothing is given the nearest turn and flagged. Dropping it would delete it
   from the hypothesis and register as a cpWER deletion — quietly *rewarding* a
   system for missing speech. The flag is what lets Stage 6 separate this rule's
   cost from real labelling errors.
3. **Contested words are counted in two buckets.** `overlap` means two speakers
   genuinely active at once; `boundary` means a word crossing between two
   disjoint turns. Merging them would bury the first: the corpus is 7.60%
   overlapped, while every turn change makes boundary words.
4. **`--diar ref` is an oracle.** It attributes with the reference RTTM, giving a
   cpWER floor where labelling is perfect by construction, so every other
   condition reads as "ASR error + what this diarizer cost". Diagnostic only —
   nothing from it is fed back to any model, and it is labelled `oracle` in
   every table.

In [ ]:
import pathlib, shutil, json

ROOT = pathlib.Path("/kaggle/input")
# Several datasets can contain a copy of the scripts: the attrib dataset is a
# snapshot of a working directory and carries whatever was current when it was
# saved. Taking the first rglob hit silently runs STALE code. Prefer the
# directory holding the most pipeline scripts, tie-broken toward a path named
# like the code dataset.
_cands = {p.parent for p in ROOT.rglob("stage4_attribute.py")}
CODE = max(_cands, key=lambda d: (len(list(d.glob("stage*.py"))),
                                  "code" in str(d).lower()))
if len(_cands) > 1:
    print("script copies found in:")
    for _c in sorted(map(str, _cands)):
        print("   ", _c, "  <-- using" if str(CODE) == _c else "")
WORK = pathlib.Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)

for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")

# Every attached dataset that carries a data/ tree is merged into one working
# copy: Stage 3's RTTMs, and one directory per ASR system from the two Stage 4a
# datasets. No audio is needed, so that dataset stays detached.
for src in sorted(ROOT.rglob("data")):
    if src.is_dir() and any((src / d).exists() for d in ("hyp", "ref", "asr")):
        shutil.copytree(src, WORK, dirs_exist_ok=True)
        print("restored", src)

# The code dataset carries ref/ at its top level, NOT inside a data/ tree, so
# the loop above misses it -- which is how ref/segments (the transcripts) can be
# absent while ref/rttm looks fine. Stage 4b never noticed; scoring needs them.
if (CODE / "ref").is_dir():
    shutil.copytree(CODE / "ref", WORK / "ref", dirs_exist_ok=True)
    print("restored", CODE / "ref")

print()
print("CODE:", CODE)
ASR = sorted(p.name for p in (WORK / "asr").glob("*") if p.is_dir())
DIAR = sorted(p.name for p in (WORK / "hyp").glob("*") if p.is_dir())
# Print the DECODE PROVENANCE, not just the count. Several attached datasets
# can carry an asr/ tree -- the attrib dataset is a snapshot of a whole working
# directory -- and copytree(dirs_exist_ok=True) lets a later one overwrite an
# earlier one. A stale words file is invisible in a file count and produces a
# full, plausible, wrong results table. `lang_locked` exists only in output
# from the multisoftmax-aware decode.
for a in ASR:
    files = sorted((WORK / "asr" / a / "words").glob("*.json"))
    d = json.loads(files[0].read_text(encoding="utf-8")) if files else {}
    locked = d.get("lang_locked", "n/a" if a.startswith("whisper") else "STALE")
    sample = " ".join(w["w"] for w in d.get("words", [])[:6])
    print(f"  asr/{a:20} {len(files):3} clips  lang_locked={locked}  {sample[:48]}")
    assert locked != "STALE", (
        f"{a}: words predate the language-mask fix. Detach sarvam-diar-attrib "
        f"(it carries an old asr/ tree that overwrites the new one) and re-attach "
        f"the current sarvam-diar-asr-indic version."
    )
for d in DIAR:
    print(f"  hyp/{d:16} {len(list((WORK / 'hyp' / d / 'rttm').glob('*.rttm'))):3} rttm")
print(f"  ref/rttm{'':12} {len(list((WORK / 'ref' / 'rttm').glob('*.rttm'))):3} rttm")
print(f"  ref/segments{'':8} {len(list((WORK / 'ref' / 'segments').glob('*.json'))):3} json"
      "   <- scoring needs these")
print()
print("ASR :", ASR)
print("DIAR:", DIAR, "+ ref (oracle)")
assert ASR, "no ASR words found -- attach the Stage 4a dataset(s)"
assert list((WORK / "ref" / "segments").glob("*.json")), (
    "no ref/segments -- Stage 4c cannot score without the reference transcripts"
)

# Built here rather than interpolated into the shell line below: IPython's {}
# expansion chokes on quotes inside the braces.
ASR_ARG, DIAR_ARG = " ".join(ASR), " ".join(DIAR)

### What must be true before running

Attribution is silent about inputs it never sees: an ASR system whose dataset is
not attached simply produces no conditions, and the run still exits 0. The cell
above prints the inventory so a missing attachment is caught here rather than
discovered as a hole in the results table.

`sortformer` has 74 of 99 RTTMs — the long clips OOM'd in Stage 3. Those 25 clips
are expected to **fail loudly** below rather than be written as empty, which is
why the script's exit code is non-zero on that condition. That is the correct
outcome, not a bug to work around: a clip with no hypothesis is not a clip where
nobody spoke.

### Language-ID fallback: one more ASR system, built from the two above

IndicConformer picks one language per clip from its own frame votes. On 13 clips
that vote lands outside the languages this task serves (11 Urdu, 2 Nepali), and
the whole clip is spelled in the wrong script: 100% WER regardless of what was
heard. `ic_lid_fallback` keeps IndicConformer's words everywhere else and takes
Whisper's on those clips.

No reference is read. The input is the model's own language decision and a fixed
list of served languages -- task configuration, not a per-clip label. CPU,
seconds, and written in Stage 4a's format so everything downstream treats it as
one more ASR system.

In [ ]:
!python stage4_fallback.py --data data

# Re-listed, not reused from the setup cell: ic_lid_fallback did not exist when
# that cell ran, and attribution only crosses the systems named in ASR_ARG.
ASR = sorted(p.name for p in (WORK / "asr").glob("*") if p.is_dir())
ASR_ARG = " ".join(ASR)
print("ASR :", ASR)

In [ ]:
!python stage4_attribute.py --asr {ASR_ARG} --diar {DIAR_ARG} ref --data data

### Read the orphan rate before believing any cpWER

If a system orphans a few percent of words, rule 2 is a footnote. If it orphans
twenty, the rule is doing heavy lifting and the writeup has to say so before
quoting a single number.

In [ ]:
import json, pathlib
import pandas as pd

rows = []
for cond in sorted((pathlib.Path("/kaggle/working/data/attrib")).glob("*")):
    mf = cond / "manifest.jsonl"
    if not mf.exists():
        continue
    # The manifest is append-only, so a retried clip has one line per attempt.
    # Collapse to the last record per clip or the failure counts multiply.
    recs = {}
    for l in mf.read_text().splitlines():
        if l.strip():
            r = json.loads(l)
            recs[r["clip_id"]] = r
    recs = list(recs.values())
    ok = [r for r in recs if r["status"] == "ok"]
    w = sum(r["n_words"] for r in ok) or 1
    asr, diar = cond.name.split("__")
    rows.append({
        "asr": asr,
        "diar": diar + (" (oracle)" if diar == "ref" else ""),
        "clips_ok": len(ok),
        "clips_fail": len(recs) - len(ok),
        "words": sum(r["n_words"] for r in ok),
        "orphan_%": round(100 * sum(r["n_orphan"] for r in ok) / w, 2),
        "overlap_%": round(100 * sum(r["n_overlap_words"] for r in ok) / w, 2),
        "boundary_%": round(100 * sum(r["n_boundary_words"] for r in ok) / w, 2),
        "mean_spk": round(sum(r["n_speakers"] for r in ok) / max(len(ok), 1), 2),
    })

df = pd.DataFrame(rows).sort_values(["asr", "diar"])
print(df.to_string(index=False))

# One clip, end to end, so the words are visibly attached to speakers.
cond = sorted(pathlib.Path("/kaggle/working/data/attrib").glob("*__ref"))[0]
clip = sorted(cond.glob("*.json"))[0]
d = json.load(open(clip, encoding="utf-8"))
print()
print(f"{d['clip_id'][:44]}  {d['lang']}  {d['n_words']} words, "
      f"{len(d['speakers'])} speakers, {d['n_orphan']} orphaned")
for spk, text in d["by_speaker"].items():
    print(f"  {spk:12} {text[:110]}")

### Which language did each clip get decoded in?

With the mask on, the language is now a **decision** taken once per clip from
frame votes, not something that emerges per token. A clip decoded in the wrong
block is ~100% WER by construction, so a handful of misidentified clips can
account for a large slice of the corpus WER — and this costs no GPU time to
check.

`clip_meta.csv` carries the reference script per clip. The diagonal should be
heavy; anything far off it is worth listening to.

In [ ]:
import json, pathlib
import pandas as pd

meta = pd.read_csv('/kaggle/working/data/ref/clip_meta.csv').set_index('clip_id')
rows = []
for a in sorted(pathlib.Path('/kaggle/working/data/asr').glob('*')):
    for f in sorted((a / 'words').glob('*.json')):
        d = json.loads(f.read_text(encoding='utf-8'))
        cid = d['clip_id']
        rows.append({
            'asr': a.name,
            'clip_id': cid,
            'detected': d.get('lang'),
            'script': meta.at[cid, 'language'] if cid in meta.index else '?',
            'hyp_words': len(d['words']),
            'ref_words': int(meta.at[cid, 'n_ref_words']) if cid in meta.index else 0,
        })

lid = pd.DataFrame(rows)
for a, g in lid.groupby('asr'):
    print('===', a, '===')
    print(pd.crosstab(g['script'], g['detected']).to_string())
    ratio = g['hyp_words'].sum() / max(g['ref_words'].sum(), 1)
    print(f'hyp/ref word ratio: {ratio:.2f}')
    print()

# Stage 4c — Scoring

Same session: scoring is CPU-only too, and it needs exactly what 4b just wrote.

Four metrics, chosen so the errors decompose instead of piling into one number.
**WER** is speaker-agnostic and must be *identical* across every diar condition
of one ASR — it is the same words either way, so if it moves, something leaked
between stages. **cpWER** is the headline. **DI-cpWER** relaxes the speaker
constraint, so `cpWER − DI-cpWER` is what wrong attribution cost, which is
precisely the quantity Stage 5 sets out to reduce. **WDER** is hand-rolled:
meeteval has no WDER.

Corpus rates are error-weighted (sum of errors / sum of reference words), with
the unweighted mean beside them — that one lets a 50 s clip outweigh a 30 min
one, and is the number people publish by accident.

In [ ]:
# rapidfuzz is not on the Kaggle image, and WDER needs it for the word
# alignment; a pure-Python DP would be minutes per clip at 3000 words.
!pip install -q meeteval rapidfuzz

The probe below matters: meeteval has moved these functions between
`meeteval.wer` and `meeteval.wer.wer.*` across releases, and DI-cpWER is recent.
`stage4_score.py` tries the known spellings and, if none match, prints what the
installed version actually exposes instead of dying on an AttributeError. If the
run fails, the list printed here is what to send back.

In [ ]:
import meeteval, meeteval.wer as mw
print("meeteval", getattr(meeteval, "__version__", "?"))
print("exposes:", sorted(n for n in dir(mw) if "error_rate" in n or n.endswith("wer")))

In [ ]:
!python stage4_score.py --data data

### Two subsets, and only one of them is a fair comparison

`sortformer` has 74 of 99 clips, and the 25 it lacks are all long ones — more
speakers, more turn changes than average. Its number over 74 clips is not
comparable to another system's over 99. The `common` table is the three-way
comparison; `all` is each condition over whatever it has.

Sanity checks worth making before believing any of it: **WER constant** across
the diar conditions of one ASR (it is the same words, so movement means a leak),
and `attribution_cost` non-negative everywhere — a large negative means the
meeteval binding is wrong, and the script says so loudly. A *small* negative is
the greedy DI-cpWER approximation and is expected.

The oracle is **not** guaranteed lowest on cpWER. When ASR error dominates, a
diarizer that merges speakers can score better than perfect diarization simply
by offering fewer ways to misattribute. If the oracle is not clearly best, that
is a statement about the ASR, not a bug in the scoring.

In [ ]:
import pathlib
import pandas as pd

csv = pathlib.Path("/kaggle/working/data/results/asr_summary.csv")
assert csv.exists(), (
    "no asr_summary.csv -- the scoring cell above did not finish. Read its "
    "output: a meeteval binding problem prints the function names the installed "
    "version exposes, and that list is the fix."
)
s = pd.read_csv(csv)
for subset in ("common", "all"):
    print(f"--- {subset} ---")
    print(s[s.subset == subset].drop(columns=["subset"]).to_string(index=False))
    print()

## Save

Strip the scripts from the working directory **before** saving. This dataset is
a snapshot of `/kaggle/working`, so it would otherwise carry its own copy of the
pipeline — and a later session's `rglob` can pick that stale copy over the code
dataset. That has already caused one silent run against old code and one against
old words.

In [ ]:
import pathlib

for _p in pathlib.Path('/kaggle/working').glob('*.py'):
    _p.unlink()
print('kept:', sorted(x.name for x in pathlib.Path('/kaggle/working').iterdir()))

Then Output tab → **New Version** of `sarvam-diar-attrib` (not a new dataset —
Stages 5 and 6 attach this by name). It carries both the attribution and
`data/results/`, so they attach one thing.